In [1]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["PYSPARK_PYTHON"] = "/home/varunadhityagb/localProjects/BigData-Intrusion/.venv/bin/python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/home/varunadhityagb/localProjects/BigData-Intrusion/.venv/bin/python3"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
spark = SparkSession.builder \
    .appName("IDS2018-binary-v2") \
    .master("spark://lattitude7420:7077") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.storageFraction", "0.3") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("UI:", spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/22 14:39:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.8
UI: http://lattitude7420:4040


### Load Test Train from HDFS

In [4]:
train = spark.read.parquet("hdfs://lattitude7420:9000/ids2018/train")
test  = spark.read.parquet("hdfs://lattitude7420:9000/ids2018/test")
train.cache()
test.cache()

26/05/22 14:40:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


DataFrame[Dst Port: float, Protocol: float, Flow Duration: float, Tot Fwd Pkts: float, Tot Bwd Pkts: float, TotLen Fwd Pkts: float, TotLen Bwd Pkts: float, Fwd Pkt Len Max: float, Fwd Pkt Len Min: float, Fwd Pkt Len Mean: float, Fwd Pkt Len Std: float, Bwd Pkt Len Max: float, Bwd Pkt Len Min: float, Bwd Pkt Len Mean: float, Bwd Pkt Len Std: float, Flow IAT Mean: float, Flow IAT Std: float, Flow IAT Max: float, Flow IAT Min: float, Fwd IAT Tot: float, Fwd IAT Mean: float, Fwd IAT Std: float, Fwd IAT Max: float, Fwd IAT Min: float, Bwd IAT Tot: float, Bwd IAT Mean: float, Bwd IAT Std: float, Bwd IAT Max: float, Bwd IAT Min: float, Fwd PSH Flags: float, Bwd PSH Flags: float, Fwd URG Flags: float, Bwd URG Flags: float, Fwd Header Len: float, Bwd Header Len: float, Fwd Pkts/s: float, Bwd Pkts/s: float, Pkt Len Min: float, Pkt Len Max: float, Pkt Len Mean: float, Pkt Len Std: float, Pkt Len Var: float, FIN Flag Cnt: float, SYN Flag Cnt: float, RST Flag Cnt: float, PSH Flag Cnt: float, ACK Fl

### Load models

In [5]:
model_binary = PipelineModel.load("hdfs://lattitude7420:9000/ids2018/models/binary")

26/05/22 14:40:11 WARN TaskSetManager: Lost task 0.0 in stage 2.0 (TID 2) (192.168.0.8 executor 4): java.io.IOException: Cannot run program "/home/varunadhityagb/localProjects/BigData-Intrusion/.venv/bin/python3": error=2, No such file or directory
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1128)
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1071)
	at org.apache.spark.api.python.PythonWorkerFactory.startDaemon(PythonWorkerFactory.scala:239)
	at org.apache.spark.api.python.PythonWorkerFactory.createThroughDaemon(PythonWorkerFactory.scala:139)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:107)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:124)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:174)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:67)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:367)
	at org.apache.spark.rdd.RDD.iter

### Generate Predictions

In [6]:
print("Binary predictions")
pred_binary = model_binary.transform(test)
pred_binary.cache()


Binary predictions


DataFrame[Dst Port: float, Protocol: float, Flow Duration: float, Tot Fwd Pkts: float, Tot Bwd Pkts: float, TotLen Fwd Pkts: float, TotLen Bwd Pkts: float, Fwd Pkt Len Max: float, Fwd Pkt Len Min: float, Fwd Pkt Len Mean: float, Fwd Pkt Len Std: float, Bwd Pkt Len Max: float, Bwd Pkt Len Min: float, Bwd Pkt Len Mean: float, Bwd Pkt Len Std: float, Flow IAT Mean: float, Flow IAT Std: float, Flow IAT Max: float, Flow IAT Min: float, Fwd IAT Tot: float, Fwd IAT Mean: float, Fwd IAT Std: float, Fwd IAT Max: float, Fwd IAT Min: float, Bwd IAT Tot: float, Bwd IAT Mean: float, Bwd IAT Std: float, Bwd IAT Max: float, Bwd IAT Min: float, Fwd PSH Flags: float, Bwd PSH Flags: float, Fwd URG Flags: float, Bwd URG Flags: float, Fwd Header Len: float, Bwd Header Len: float, Fwd Pkts/s: float, Bwd Pkts/s: float, Pkt Len Min: float, Pkt Len Max: float, Pkt Len Mean: float, Pkt Len Std: float, Pkt Len Var: float, FIN Flag Cnt: float, SYN Flag Cnt: float, RST Flag Cnt: float, PSH Flag Cnt: float, ACK Fl

### THRESHOLD TUNING

In [12]:
thresholds = [0.5, 0.4, 0.3, 0.2, 0.1]

print(f"{'Threshold':>10} {'TP':>10} {'FP':>10} {'FN':>10} {'TN':>10} {'Recall':>8} {'Precision':>10} {'F1':>8}")
print("-" * 80)

for t in thresholds:
    # re-run transform with threshold set on the RF stage
    rf_stage = model_binary.stages[-1]
    rf_stage.setThresholds([1.0 - t, t])  # [benign_threshold, attack_threshold]
    
    pred_t = model_binary.transform(test)
    
    tp = pred_t.filter((F.col("prediction") == 1.0) & (F.col("binary_label") == 1.0)).count()
    fp = pred_t.filter((F.col("prediction") == 1.0) & (F.col("binary_label") == 0.0)).count()
    fn = pred_t.filter((F.col("prediction") == 0.0) & (F.col("binary_label") == 1.0)).count()
    tn = pred_t.filter((F.col("prediction") == 0.0) & (F.col("binary_label") == 0.0)).count()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    prec   = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1     = 2 * prec * recall / (prec + recall) if (prec + recall) > 0 else 0

    print(f"{t:>10.1f} {tp:>10,} {fp:>10,} {fn:>10,} {tn:>10,} {recall:>8.4f} {prec:>10.4f} {f1:>8.4f}")

# reset threshold back to default after loop
model_binary.stages[-1].setThresholds([0.5, 0.5])

 Threshold         TP         FP         FN         TN   Recall  Precision       F1
--------------------------------------------------------------------------------


       0.5    415,548      4,574     30,060  1,356,633   0.9325     0.9891   0.9600


       0.4    417,123     10,080     28,485  1,351,127   0.9361     0.9764   0.9558


       0.3    418,917     16,759     26,691  1,344,448   0.9401     0.9615   0.9507


       0.2    420,586     26,599     25,022  1,334,608   0.9438     0.9405   0.9422


[Stage 80:==================================================>     (10 + 1) / 11]

       0.1    421,672     45,525     23,936  1,315,682   0.9463     0.9026   0.9239


RandomForestClassificationModel: uid=RandomForestClassifier_d95a74fb9eb2, numTrees=20, numClasses=2, numFeatures=81

In [13]:
model_binary.stages[-1].setThresholds([0.5, 0.5])

fn_rows = pred_binary.filter(
    (F.col("prediction") == 0.0) & (F.col("binary_label") == 1.0)
)
fn_rows.groupBy("attack_type").count().orderBy("count", ascending=False).show()

[Stage 83:==============================================>          (9 + 2) / 11]

+--------------+-----+
|   attack_type|count|
+--------------+-----+
| Infilteration|29207|
|          DDoS|  494|
|     WebAttack|  169|
|           Bot|  129|
|           DoS|   60|
|SSH-Bruteforce|    1|
+--------------+-----+



In [14]:
spark.stop()